In [0]:
import logging
import yaml

dbutils.widgets.text("environment", "dev")
dbutils.widgets.text("config_path", "")

environment = dbutils.widgets.get("environment")
CONFIG_PATH = dbutils.widgets.get("config_path")

logger = logging.getLogger("table_maintenance")
logger.setLevel(logging.INFO)

try:
    with open(CONFIG_PATH) as f:
        full_config = yaml.safe_load(f)

    config = full_config[environment]

    catalog = config["catalog"]
    silver_schema = config["silver_schema"]

    consumption_table = f"{catalog}.{silver_schema}.{config['consumption']['target_table']}"
    prices_table = f"{catalog}.{silver_schema}.{config['prices']['target_table']}"

    logger.info(f"[{environment}] Config loaded. Tables: {consumption_table}, {prices_table}")

except Exception as e:
    logger.error(f"Failed to load config from {CONFIG_PATH} for environment '{environment}': {e}")
    raise

In [0]:
import logging

logger = logging.getLogger("table_maintenance")
logger.setLevel(logging.INFO)

tables_to_maintain = [consumption_table, prices_table]

for table in tables_to_maintain:
    try:
        optimize_result = spark.sql(f"OPTIMIZE {table}")
        stats = optimize_result.collect()[0].asDict()
        logger.info(f"OPTIMIZE completed for {table}: {stats}")

        spark.sql(f"VACUUM {table}")
        logger.info(f"VACUUM completed for {table}")

    except Exception as e:
        logger.error(f"Maintenance failed for {table}: {e}")
        raise